In [1]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType


/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from scipy.stats import ttest_rel

In [3]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *
from pro_gen2_lora import *

In [4]:
with open('/Users/johnhutchens/Desktop/Practicum/Data/Domainome/dict_domainome_uniprot.pkl',
           'rb') as f:
    dict_uniprot = pickle.load(f)

In [ ]:
# path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'

# with open(path+"dict_PF00030.pkl", "rb") as f:
#     dict_PF00030 = pickle.load(f)

In [ ]:
# keys = dict_uniprot.keys()
# keys

dict_keys(['P00519', 'O94929', 'Q9H2P0', 'Q8N157', 'Q15699', 'O95076', 'Q9H161', 'P10275', 'P16157', 'Q01484', 'Q12955', 'Q68DC2', 'Q7Z6G8', 'P10398', 'Q8N1W1', 'Q68CP9', 'Q6ZSG1', 'Q96QS3', 'A6NK59', 'Q9ULZ3', 'Q8IZT6', 'Q8WWH4', 'O15265', 'O75366', 'O95817', 'Q9UL15', 'Q9UIF8', 'P51451', 'Q12830', 'P15056', 'O60885', 'Q06187', 'Q9NPB3', 'Q96GE6', 'O75808', 'P22681', 'P45973', 'Q8IX12', 'P41002', 'Q9Y5K6', 'Q12873', 'Q14839', 'Q8TDI0', 'Q9P2D1', 'Q9UHD4', 'P61024', 'Q8WXI2', 'P10589', 'Q7Z5Q1', 'P05813', 'P53673', 'P53674', 'P43320', 'Q8N1P7', 'P11844', 'P07316', 'P07315', 'P07320', 'O43186', 'P22914', 'O75534', 'O14936', 'P0A9X9', 'P41016', 'P32081', 'P36075', 'Q8IWT3', 'P39880', 'O14529', 'O94830', 'Q9BTC0', 'Q9UBS4', 'Q5F1R6', 'Q92796', 'P78352', 'Q8NFW5', 'Q9Y222', 'P25685', 'O75953', 'Q7Z6W7', 'Q96KC8', 'Q99543', 'Q9H3Z4', 'Q99615', 'Q6XZF7', 'Q96FX2', 'P55265', 'Q9Y4J8', 'O14640', 'Q92997', 'P78545', 'Q9UKW6', 'P60002', 'Q09472', 'Q12929', 'Q96RT1', 'P11308', 'Q8IV48', 'O95718',

In [ ]:
# for key in keys:
#     seq = dict_uniprot[key]
#     seq_len = len(seq)
#     if seq_len < 1000:
#         print(key, seq_len)

O94929 683
Q15699 326
O95076 343
Q9H161 411
P10275 920
Q68DC2 871
P10398 606
Q6ZSG1 346
Q96QS3 562
A6NK59 587
Q9ULZ3 195
Q8WWH4 475
O15265 892
O75366 819
O95817 575
Q9UL15 447
P51451 505
P15056 766
Q06187 659
Q9NPB3 220
Q96GE6 196
P22681 906
P45973 191
P41002 786
Q9Y5K6 639
Q9UHD4 219
P61024 79
P10589 423
Q7Z5Q1 589
P05813 215
P53673 196
P53674 252
P43320 205
P11844 174
P07316 175
P07315 174
P07320 174
O43186 299
P22914 178
O75534 798
O14936 926
P0A9X9 70
P41016 66
P32081 67
P36075 443
O94830 711
Q9UBS4 358
Q5F1R6 531
Q92796 817
P78352 724
Q8NFW5 382
Q9Y222 760
P25685 340
O75953 348
Q7Z6W7 309
Q96KC8 554
Q99543 621
Q9H3Z4 198
Q99615 494
Q96FX2 82
Q9Y4J8 743
O14640 695
Q92997 716
P78545 371
Q9UKW6 265
P60002 83
Q12929 822
P11308 479
Q8IV48 349
O95718 433
P14921 441
Q13158 208
Q9NW38 375
Q86XK2 927
Q8NEZ5 403
Q86WN1 690
A0PJY2 475
Q13642 323
Q14192 279
Q01543 452
Q06787 632
Q96AE4 644
Q92945 711
Q96I24 572
P06241 537
Q99501 681
Q06546 454
P04150 777
Q5VTD9 330
Q99684 422
Q8NEA6 775
P6299

Create testing data from PF00030 to test model trained on P07316_PF00030_87

In [12]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')
blob = bucket.blob('SupplementaryTable2.txt')

df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [77]:
df[df['uniprot_ID'] == 'Q96QS3']

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
492797,Q96QS3_PF00046_330,Q96QS3,*YRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,*,True,1.0,1.0,1.0,0.0,0.0,1.0,1.000000,0.079746,0.183649,-0.155440,1.878233,225
492798,Q96QS3_PF00046_330,Q96QS3,AYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,A,False,10.0,16.0,7.0,94.0,64.0,12.0,11.000000,0.127975,0.027376,0.337811,0.279982,225
492799,Q96QS3_PF00046_330,Q96QS3,CYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,C,False,14.0,30.0,15.0,53.0,8.0,30.0,19.666670,0.080829,0.023973,-0.144370,0.245173,225
492800,Q96QS3_PF00046_330,Q96QS3,DYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,D,False,9.0,3.0,7.0,69.0,0.0,0.0,6.333333,0.141550,0.052804,0.476643,0.540043,225
492801,Q96QS3_PF00046_330,Q96QS3,EYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,E,False,12.0,10.0,6.0,15.0,25.0,8.0,9.333333,0.096457,0.030725,0.015470,0.314234,225
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493913,Q96QS3_PF00046_330,Q96QS3,SYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,S,False,23.0,33.0,41.0,50.0,102.0,49.0,32.333330,0.103085,0.017228,0.083250,0.176199,225
493914,Q96QS3_PF00046_330,Q96QS3,TYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,T,False,1.0,7.0,5.0,0.0,25.0,2.0,4.333333,0.101910,0.047215,0.071232,0.482883,225
493915,Q96QS3_PF00046_330,Q96QS3,VYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,V,False,18.0,24.0,27.0,8.0,28.0,13.0,23.000000,0.068358,0.021502,-0.271914,0.219908,225
493916,Q96QS3_PF00046_330,Q96QS3,WYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVW...,R,330.0,W,False,8.0,9.0,12.0,0.0,4.0,6.0,9.666667,0.056692,0.040841,-0.391217,0.417693,225


In [4]:
make_mutation_csv('Q96QS3_PF00046_330')

In [5]:
df_mutation = pd.read_csv('mutation_Q96QS3_PF00046_330.csv')
df_mutation['wt_seq'].iloc[0]

'RYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVWFQNRRAKWRK'

Load base model

In [10]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


# Fine tune model using 5 epochs, learning rate 1e-3, CausalLM, num_samples = i

## Q96QS3

In [8]:
seq = dict_uniprot['Q96QS3']
dom_seq = df_mutation['wt_seq'].iloc[0]
dom_pos = 330

print(seq[dom_pos-1:dom_pos-1+len(dom_seq)])
print(dom_seq)

RYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVWFQNRRAKWRK
RYRTTFTSYQLEELERAFQKTHYPDVFTREELAMRLDLTEARVQVWFQNRRAKWRK


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)


path = 'mutation_Q96QS3_PF00046_330.csv'
path_name = os.path.splitext(path)[0]
path_list = path_name.split("_")

dom_pos = int(path_list[3])

df_mutation = pd.read_csv(path)
                          
protein_seq = dict_uniprot[path_list[1]]
dom_seq = df_mutation['wt_seq'].iloc[0]

fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(dom_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss

eps = 3
lr = 1e-3

i = 3
print(f"{i} num_samples")    
model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, dom_seq, dom_pos,
                                        exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                        k=0.8, num_samples=i, print_info=False)

print(f"Training loss = {train_losses}")
print(f"Validation loss = {val_losses}")


    

In [11]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)


path = 'mutation_Q96QS3_PF00046_330.csv'
path_name = os.path.splitext(path)[0]
path_list = path_name.split("_")

dom_pos = int(path_list[3])

df_mutation = pd.read_csv(path)
                          
protein_seq = dict_uniprot[path_list[1]]
dom_seq = df_mutation['wt_seq'].iloc[0]

fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(dom_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss

eps = 5
lr = 1e-3

for i in range(1,11):    
    print(f"{i} num_samples")    
    model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                            lora_config, protein_seq, dom_seq, dom_pos,
                                            exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                            k=0.8, num_samples=i, print_info=False)

    print(f"Training loss = {train_losses}")
    print(f"Validation loss = {val_losses}")


    

1 num_samples
Training loss = [0.0, 0.0, 0.0, 0.0, 0.0]
Validation loss = [0.0, 0.0, 0.0, 0.0, 5.960464477539063e-08]
2 num_samples
Training loss = [0.44678398966789246, 0.8124198913574219, 0.9023404121398926, 0.025976896286010742, 2.3015711307525635]
Validation loss = [2.396275520324707, 0.49647092819213867, 5.072346210479736, 0.09067630767822266, 1.6735495328903198]
3 num_samples
Training loss = [0.31386494636535645, 2.3899118900299072, 0.046358466148376465, 0.3360036611557007, 1.9820518493652344]
Validation loss = [1.8061614036560059, 7.974296569824219, 0.8480029106140137, 1.179115891456604, 2.0098884105682373]
4 num_samples
Training loss = [0.7870529890060425, 1.363101840019226, 1.0149571895599365, 2.37522554397583, 7.752183437347412]
Validation loss = [4.025030136108398, 4.12885046005249, 1.561354160308838, 4.729152679443359, 0.09729072451591492]
5 num_samples
Training loss = [3.2834441661834717, 0.9957538843154907, 1.6077880859375, 6.782663822174072, 1.2179815769195557]
Validatio

In [19]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
# key = keys[0]
# protein_seq = dict_PF00030[key]['wt_seq']
path = 'mutation_Q96QS3_PF00046_330.csv'
path_name = os.path.splitext(path)[0]
path_list = path_name.split("_")

dom_pos = int(path_list[3])

df_mutation = pd.read_csv(path)
                          
protein_seq = dict_uniprot[path_list[1]]
dom_seq = df_mutation['wt_seq'].iloc[0]

fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(dom_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss

eps = 5
lr = 1e-3

for i in range(1,11):    
    print(f"{i} num_samples")    
    model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                            lora_config, protein_seq, dom_seq, dom_pos,
                                            exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                            k=0.8, num_samples=i, print_info=False)

    print(f"Training loss = {train_losses}")
    print(f"Validation loss = {val_losses}")


    

1 num_samples
Training loss = [0.0, 0.0, 0.0, 5.960464477539063e-08, 0.0]
Validation loss = [0.0, 0.0, 0.0, 0.0, 8.0108642578125e-05]
2 num_samples
Training loss = [0.10630261898040771, 1.8596649169921875e-05, 1.500455617904663, 0.049013376235961914, 0.041722655296325684]
Validation loss = [0.32374879717826843, 1.2958571910858154, 0.0001766681671142578, 2.0265579223632812e-06, 0.8427600860595703]
3 num_samples
Training loss = [0.005398273468017578, 1.6399749517440796, 4.149914264678955, 1.034926176071167, 1.2649248838424683]
Validation loss = [0.3239707350730896, 0.4935464859008789, 2.4308297634124756, 2.4311411380767822, 0.0348464660346508]
4 num_samples
Training loss = [5.839535713195801, 5.602147102355957, 0.7592688202857971, 1.104435920715332, 1.119797945022583]
Validation loss = [2.4305853843688965, 1.1938683986663818, 1.6180856227874756, 0.879270613193512, 0.904641330242157]
5 num_samples
Training loss = [1.5942933559417725, 1.297430396080017, 0.9786985516548157, 4.65838766098022

In [ ]:
model_llrs = []

    for key in dict_PF00030.keys():
        seq = dict_PF00030[key]['wt_seq']
        lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
        model_llrs.append(llr)
            
    model_llrs_dict = {}
    for k, v in zip(keys, model_llrs):
        model_llrs_dict[k] = v

    SpearmanBase = []
    SpearmanLoRA = []

    for key in keys:
        base_llr = dict_PF00030[key]['llr_pg2']
        lora_llr = model_llrs_dict[key]
        dms_mat = dict_PF00030[key]['DMS_mat']

        # print(len(base_llr), len(lora_llr), len(dms_mat))


        spb = spearman_ignore_nan(base_llr, dms_mat)
        spl = spearman_ignore_nan(lora_llr, dms_mat)

        SpearmanBase.append(spb[0])
        SpearmanLoRA.append(spl[0])

    for i in range(len(SpearmanBase)):
        spb = SpearmanBase[i]
        spl = SpearmanLoRA[i]
        print(f"Base: {spb}")
        print(f"LoRA: {spl}")
        print()

In [ ]:
# Q8N1P7_seq = 'MEEAGGPMARAKARVVSATLTWRQRPPTQEEIKHGFHKVSLVSGAQMEAPQKEMFEFSRR\
# EEVEVNGFATQEEETVNCQGPRDTAGSKNFQSHGPIFSKKYIPPPKEKRPEGRLKEAVDQ\
# SDGSRQAPRTEPPCVGAMARTELLVPLPGPREPSPHPGVGLTSGSSRSLEEYRVTRTVRT\
# TTVVGGHVDRRMSSSVTVRPVSSGEALPRGRQVSRMVPPVVVGSPPGSPSRSQAVKVLSN\
# LVPAGHSPPASHLPRPTAGGPRSTGLGSTVGAALRQLPETGTAELKDSSALASTGIPASA\
# HLPKNQDAPAACPDRDQGRAPDARACELWQVLGAPSSTELPLQTSQGQASVPSSPRLETH\
# VPSPGLTHPAKQPVVPTHPGARLTPLVLPPKKKDGPVDPPAATVLPMVRSEHVTVPGQPP\
# APSTTRRKDVPSPGGLSAPSSPRNKFVQNSENVPVLPFTQREVVKGPGAPAASSPTRKEV\
# VQGSSASAASSPTWKEVVKGPGAPAASSPTQKEVVQGSSAPAALFPTWKEVVKGPGAPDA\
# SFPTWKEVVKGPGAPAASSPTQKEVVQGSGAPAALSTTPKEVVKGPGAPAASSPTQKEVV\
# KGPCAPAASSPTQKEVVQGSGAPAALSPKSTEVVQGPKGSSSIQKEAVQGIAGSLAPPLT\
# KEETVQGPIAPATSLPKQDKGVQDSEGSPISSLTQKEVVQDPDALPAPSSSVDRVSPSPG\
# GTPAPVPTGAEASTESQLVSDPTEGKTCTETSREEDEVALAADLEIFLDTLRSMEPPEIL\
# RTHRLPRAPRSSYLSMYATLPAIEEDQLGPWVLGPGPQEVPSLEEKEEEEEEEPENPYLS\
# DDEKLQRRQEKAGPSPSRDLHPARPTQVSCSPLEMMKKHVAGTKGPHSELGLELQGGSRP\
# TSRLGGSLLFGSLVPTAKEASTPEPLGTKLSALLPHGAPGLRKVPGQLPLLCSERSSPTE\
# KLACSLPLEGWSPALKTQGKLNTRPGKVIFFSESGCQGSGREVWGDIVDASGWAPVASIR\
# VVRGCWVLYEEPEFRGQKLVLPEGDMELRTPGTKWSPQGIGSLRRVVWDYSTPEISLFSE\
# EGLKGEQVKLTEALKNSQGLEKPLQVASATVSAGLWLLYPKPLFEDTPYILEPGEYPTSE\
# AWGTSDPSVGSLKPMRLGCPSVEKPGEPRAVVYEAPGFQGRSWEVSRDIYNLQQPEDSQS\
# PHLASVGSLRVLGGCWVGYEKEGFRGHQYLLEEGEYPDWSHWGGYDELLTSLRVIRTDFG\
# DPAVVLFEAMDFEGHGVEVSKALPDVELVQHGPSTQAIHVLSGVWVAYQEVGFSGEQYVL\
# EKGVYRNCEDWGAGNSTLASLQPVLQVGEHDLHFVSKIQLFSRPDFLGDHFSFEDDQAAL\
# PASFRPQSCRVHGGSWILFDETNFEGDQHILSEGEFPTLTAMGCLASTVLGSLQKVSLHF\
# SEPSIFLYGLECFEGKEIELSREVRSLQAEGFNNHVLSVRIKGGIWVLCEHSDFRGRQWL\
# VGSCEITNWLTYSGTQRVGSLYPIKQRRVYFRLWNAALGGFLAVPDHVEDMKAGRVVVAD\
# PQAGGSCIWYYEDGLLKNQMAPTMSLQVIGPPSPGSKVVLWAESRLPRQTWSISESGHIC\
# SQMFEGQILDVKGGRGYDRDHVVLWEPDEDRASQIWTIHVL'

In [21]:
# sequence is too long to input without batching
Q8N1P7_seq_638 = Q8N1P7_seq[637:] 

In [22]:
dom_seq = dict_PF00030[key]['wt_seq']
dom_seq

'IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETNFEGDQHILSEGEFPTLTAMGCLASTVLGSLQK'

In [25]:
key_list = key.split("_")
dom_pos = int(key_list[-1])
dom_len = len(dom_seq)
print(dom_seq)
print(Q8N1P7_seq_638[dom_pos-1-637:dom_pos-1-637+ dom_len])


IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETNFEGDQHILSEGEFPTLTAMGCLASTVLGSLQK
IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETNFEGDQHILSEGEFPTLTAMGCLASTVLGSLQK


In [29]:
dom_len

78

In [14]:
for i in range(len(Q8N1P7_seq)):
    x = Q8N1P7_seq[i]
    if not x.isalpha():
        print(i, ord(x))

In [26]:
protein_seq = Q8N1P7_seq_638
dom_seq = dict_PF00030[key]['wt_seq']

In [27]:
protein_seq[dom_pos-1-637:dom_pos-1-637+len(dom_seq)]

'IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETNFEGDQHILSEGEFPTLTAMGCLASTVLGSLQK'

# Fine tune model using 5 epochs, learning rate 1e-3, CausalLM, num_samples = i

In [68]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
# key = keys[0]
# protein_seq = dict_PF00030[key]['wt_seq']
protein_seq = Q8N1P7_seq_638
dom_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation_Q8N1P7.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(dom_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss

eps = 2
lr = 1e-3

for i in range(1,6):    
    print(f"{i} num_samples")    
    model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                            lora_config, protein_seq, dom_seq, dom_pos-637,
                                            exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                            k=0.8, num_samples=i, print_info=False)

    print(f"Training loss = {train_losses}")
    print(f"Validation loss = {val_losses}")

    model.eval()

    model_llrs = []

    for key in dict_PF00030.keys():
        seq = dict_PF00030[key]['wt_seq']
        lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
        model_llrs.append(llr)
            
    model_llrs_dict = {}
    for k, v in zip(keys, model_llrs):
        model_llrs_dict[k] = v

    SpearmanBase = []
    SpearmanLoRA = []

    for key in keys:
        base_llr = dict_PF00030[key]['llr_pg2']
        lora_llr = model_llrs_dict[key]
        dms_mat = dict_PF00030[key]['DMS_mat']

        # print(len(base_llr), len(lora_llr), len(dms_mat))


        spb = spearman_ignore_nan(base_llr, dms_mat)
        spl = spearman_ignore_nan(lora_llr, dms_mat)

        SpearmanBase.append(spb[0])
        SpearmanLoRA.append(spl[0])

    for i in range(len(SpearmanBase)):
        spb = SpearmanBase[i]
        spl = SpearmanLoRA[i]
        print(f"Base: {spb}")
        print(f"LoRA: {spl}")
        print()

1 num_samples
Training loss = [0.0, 5.960464477539063e-08]
Validation loss = [0.0, 0.0]
Base: 0.4172571483248994
LoRA: 0.4200202918827421

Base: 0.33897700677632076
LoRA: 0.3382265714345654

Base: 0.36205722660833856
LoRA: 0.3596900488536491

Base: 0.21845045525531706
LoRA: 0.2186497888523822

Base: 0.3662332108928841
LoRA: 0.3653208029329743

Base: 0.2790765656023974
LoRA: 0.27991358429466645

Base: 0.43572135931376255
LoRA: 0.43596615363684726

Base: 0.31101513899232713
LoRA: 0.30996897848359917

Base: 0.26475198062516286
LoRA: 0.263453199156219

Base: 0.3800703713136749
LoRA: 0.37651076334983474

Base: 0.184950854040034
LoRA: 0.18610762750894275

Base: 0.35140039018183716
LoRA: 0.35212952629676025

2 num_samples
Training loss = [0.002660810947418213, 0.0033376216888427734]
Validation loss = [1.722957730293274, 1.4351871013641357]
Base: 0.4172571483248994
LoRA: 0.4071980923982597

Base: 0.33897700677632076
LoRA: 0.34869288078169874

Base: 0.36205722660833856
LoRA: 0.3672981351774683


In [66]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
# key = keys[0]
# protein_seq = dict_PF00030[key]['wt_seq']
protein_seq = Q8N1P7_seq_638
dom_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation_Q8N1P7.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(dom_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss

eps = 6
lr = 1e-3

for i in range(11,21):    
    print(f"{i} num_samples")    
    model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                            lora_config, protein_seq, dom_seq, dom_pos-637,
                                            exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                            k=0.8, num_samples=i, print_info=False)

    print(f"Training loss = {train_losses}")
    print(f"Validation loss = {val_losses}")

    model.eval()

    model_llrs = []

    for key in dict_PF00030.keys():
        seq = dict_PF00030[key]['wt_seq']
        lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
        model_llrs.append(llr)
            
    model_llrs_dict = {}
    for k, v in zip(keys, model_llrs):
        model_llrs_dict[k] = v

    SpearmanBase = []
    SpearmanLoRA = []

    for key in keys:
        base_llr = dict_PF00030[key]['llr_pg2']
        lora_llr = model_llrs_dict[key]
        dms_mat = dict_PF00030[key]['DMS_mat']

        # print(len(base_llr), len(lora_llr), len(dms_mat))


        spb = spearman_ignore_nan(base_llr, dms_mat)
        spl = spearman_ignore_nan(lora_llr, dms_mat)

        SpearmanBase.append(spb[0])
        SpearmanLoRA.append(spl[0])

    for i in range(len(SpearmanBase)):
        spb = SpearmanBase[i]
        spl = SpearmanLoRA[i]
        print(f"Base: {spb}")
        print(f"LoRA: {spl}")
        print()

11 num_samples
Training loss = [4.823653697967529, 2.044893503189087, 2.6810989379882812, 2.344421148300171, 1.5567145347595215, 1.4741097688674927]
Validation loss = [5.941891670227051, 1.9647126197814941, 2.710167407989502, 1.3045812845230103, 1.8537425994873047, 1.6352821588516235]
Base: 0.4172571483248994
LoRA: 0.3194010709283491

Base: 0.33897700677632076
LoRA: 0.21582262422346918

Base: 0.36205722660833856
LoRA: 0.19874357084762087

Base: 0.21845045525531706
LoRA: 0.10945244092250409

Base: 0.3662332108928841
LoRA: 0.2681226304269601

Base: 0.2790765656023974
LoRA: 0.2325521097432423

Base: 0.43572135931376255
LoRA: 0.2583654739568492

Base: 0.31101513899232713
LoRA: 0.20352812561188768

Base: 0.26475198062516286
LoRA: 0.16262635087199498

Base: 0.3800703713136749
LoRA: 0.27925645591807025

Base: 0.184950854040034
LoRA: 0.1532794625143644

Base: 0.35140039018183716
LoRA: 0.1269484823195859

12 num_samples
Training loss = [3.4961612224578857, 2.3157222270965576, 2.1488969326019287

In [64]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
# key = keys[0]
# protein_seq = dict_PF00030[key]['wt_seq']
protein_seq = Q8N1P7_seq_638
dom_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation_Q8N1P7.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(dom_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss

eps = 5
lr = 1e-3

for i in range(6,11):    
    print(f"{i} num_samples")    
    model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                            lora_config, protein_seq, dom_seq, dom_pos-637,
                                            exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                            k=0.8, num_samples=i, print_info=False)

    print(f"Training loss = {train_losses}")
    print(f"Validation loss = {val_losses}")

    model.eval()

    model_llrs = []

    for key in dict_PF00030.keys():
        seq = dict_PF00030[key]['wt_seq']
        lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
        model_llrs.append(llr)
            
    model_llrs_dict = {}
    for k, v in zip(keys, model_llrs):
        model_llrs_dict[k] = v

    SpearmanBase = []
    SpearmanLoRA = []

    for key in keys:
        base_llr = dict_PF00030[key]['llr_pg2']
        lora_llr = model_llrs_dict[key]
        dms_mat = dict_PF00030[key]['DMS_mat']

        # print(len(base_llr), len(lora_llr), len(dms_mat))


        spb = spearman_ignore_nan(base_llr, dms_mat)
        spl = spearman_ignore_nan(lora_llr, dms_mat)

        SpearmanBase.append(spb[0])
        SpearmanLoRA.append(spl[0])

    for i in range(len(SpearmanBase)):
        spb = SpearmanBase[i]
        spl = SpearmanLoRA[i]
        print(f"Base: {spb}")
        print(f"LoRA: {spl}")
        print()

6 num_samples
Training loss = [1.48314368724823, 1.862766146659851, 1.718071460723877, 1.7941755056381226, 0.6803188323974609]
Validation loss = [2.290888547897339, 3.420947790145874, 3.615676164627075, 1.6394394636154175, 1.8961377143859863]
Base: 0.4172571483248994
LoRA: 0.4122540073436324

Base: 0.33897700677632076
LoRA: 0.31726315530293564

Base: 0.36205722660833856
LoRA: 0.3237659546221569

Base: 0.21845045525531706
LoRA: 0.20346757423133038

Base: 0.3662332108928841
LoRA: 0.33067489898203134

Base: 0.2790765656023974
LoRA: 0.2577786507989442

Base: 0.43572135931376255
LoRA: 0.40451188053314674

Base: 0.31101513899232713
LoRA: 0.2826975637685487

Base: 0.26475198062516286
LoRA: 0.26468151701128534

Base: 0.3800703713136749
LoRA: 0.33897624679060234

Base: 0.184950854040034
LoRA: 0.2321741494813657

Base: 0.35140039018183716
LoRA: 0.3663230606329597

7 num_samples
Training loss = [6.054157733917236, 5.99202299118042, 1.1524244546890259, 1.6777516603469849, 2.014777183532715]
Valida

In [63]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
# key = keys[0]
# protein_seq = dict_PF00030[key]['wt_seq']
protein_seq = Q8N1P7_seq_638
dom_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation_Q8N1P7.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(dom_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss

eps = 5
lr = 1e-3

for i in range(6):    
    print(f"{i} num_samples")    
    model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                            lora_config, protein_seq, dom_seq, dom_pos-637,
                                            exp_tensor, loss, lrate=lr, num_epochs=eps, 
                                            k=0.8, num_samples=i, print_info=False)

    print(f"Training loss = {train_losses}")
    print(f"Validation loss = {val_losses}")

    model.eval()

    model_llrs = []

    for key in dict_PF00030.keys():
        seq = dict_PF00030[key]['wt_seq']
        lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
        model_llrs.append(llr)
            
    model_llrs_dict = {}
    for k, v in zip(keys, model_llrs):
        model_llrs_dict[k] = v

    SpearmanBase = []
    SpearmanLoRA = []

    for key in keys:
        base_llr = dict_PF00030[key]['llr_pg2']
        lora_llr = model_llrs_dict[key]
        dms_mat = dict_PF00030[key]['DMS_mat']

        # print(len(base_llr), len(lora_llr), len(dms_mat))


        spb = spearman_ignore_nan(base_llr, dms_mat)
        spl = spearman_ignore_nan(lora_llr, dms_mat)

        SpearmanBase.append(spb[0])
        SpearmanLoRA.append(spl[0])

    for i in range(len(SpearmanBase)):
        spb = SpearmanBase[i]
        spl = SpearmanLoRA[i]
        print(f"Base: {spb}")
        print(f"LoRA: {spl}")
        print()

0 num_samples
Training loss = [nan, nan, nan, nan, nan]
Validation loss = [nan, nan, nan, nan, nan]
Base: 0.4172571483248994
LoRA: 0.4172571483248994

Base: 0.33897700677632076
LoRA: 0.33897700677632076

Base: 0.36205722660833856
LoRA: 0.36205722660833856

Base: 0.21845045525531706
LoRA: 0.21845045525531706

Base: 0.3662332108928841
LoRA: 0.3662332108928841

Base: 0.2790765656023974
LoRA: 0.2790765656023974

Base: 0.43572135931376255
LoRA: 0.43572135931376255

Base: 0.31101513899232713
LoRA: 0.31101513899232713

Base: 0.26475198062516286
LoRA: 0.26475198062516286

Base: 0.3800703713136749
LoRA: 0.3800703713136749

Base: 0.184950854040034
LoRA: 0.184950854040034

Base: 0.35140039018183716
LoRA: 0.35140039018183716

1 num_samples
Training loss = [0.0, 5.960464477539063e-08, 0.0, 0.0, 2.9802322387695312e-08]
Validation loss = [0.0, 0.0, 0.0, -5.960464477539063e-08, 0.0]
Base: 0.4172571483248994
LoRA: 0.41681175931870845

Base: 0.33897700677632076
LoRA: 0.33750953302419573

Base: 0.3620572

In [62]:
model_llrs_dict

{'P05813_PF00030_31': {'P05813_PF00030_31': array([[-2.3243103 , -2.1948853 , -3.653374  , ..., -2.5431519 ,
           0.        , -0.8266754 ],
         [-2.7361603 , -3.696243  , -4.247452  , ..., -3.1028137 ,
          -4.343384  , -4.3347015 ],
         [-2.5555954 , -4.323059  , -4.995529  , ..., -1.0913925 ,
          -4.8959503 , -3.9201508 ],
         ...,
         [-2.2997131 ,  0.        , -2.1996994 , ..., -2.8872833 ,
          -5.7140503 , -2.6406937 ],
         [-1.256073  , -0.51147467, -4.1340866 , ..., -2.969635  ,
          -5.9735794 , -3.5961683 ],
         [ 0.        , -4.7483826 , -3.8704226 , ..., -3.2694397 ,
          -7.473404  , -5.9514923 ]], shape=(89, 20), dtype=float32)},
 'P07315_PF00030_4': {'P07315_PF00030_4': array([[-2.9095383 , -4.043434  , -5.5880203 , ..., -0.898636  ,
          -4.9041824 , -4.347534  ],
         [-1.4061891 , -2.340561  , -2.8435516 , ...,  0.05846405,
          -2.8015442 , -2.044876  ],
         [-2.7248535 , -2.846962  , -5

In [55]:
model.eval()
ft_model_name = 'llr_pg2_lora_eps5_lr1eneg3_PF00030_CLM_ns20_ls'

In [56]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key][ft_model_name] = llr

In [57]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]][ft_model_name]

array([[-0.12954712, -0.17646027, -0.17567468, ..., -0.11204529,
         0.        , -0.05371869],
       [-0.07632446, -0.11833143, -0.09645081, ..., -0.06785583,
        -0.03694153, -0.10053253],
       [-0.08197021, -0.19789124, -0.16202545, ..., -0.02725983,
        -0.10442352, -0.12155151],
       ...,
       [-0.15674591,  0.        ,  0.02332306, ..., -0.24409485,
        -0.37688446, -0.23865509],
       [-0.04821789,  0.01367182, -0.17938995, ..., -0.2633667 ,
        -0.30467224, -0.16089606],
       [ 0.        , -0.46064758, -0.18757653, ..., -0.28883362,
        -0.4174347 , -0.41925812]], shape=(89, 20), dtype=float32)

In [58]:
SpearmanBase = []
SpearmanLoRA = []

for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key][ft_model_name]
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    SpearmanBase.append(spb[0])
    SpearmanLoRA.append(spl[0])

In [59]:
for i in range(len(SpearmanBase)):
    spb = SpearmanBase[i]
    spl = SpearmanLoRA[i]
    print(f"Base: {spb}")
    print(f"LoRA: {spl}")
    print()

Base: 0.4172571483248994
LoRA: 0.42515251745112353

Base: 0.33897700677632076
LoRA: 0.3441334206894713

Base: 0.36205722660833856
LoRA: 0.3659536908914134

Base: 0.21845045525531706
LoRA: 0.21934781787981877

Base: 0.3662332108928841
LoRA: 0.36690866810284223

Base: 0.2790765656023974
LoRA: 0.2803262171441053

Base: 0.43572135931376255
LoRA: 0.43921861362107684

Base: 0.31101513899232713
LoRA: 0.31109128598282504

Base: 0.26475198062516286
LoRA: 0.2740006021334016

Base: 0.3800703713136749
LoRA: 0.38514390180359925

Base: 0.184950854040034
LoRA: 0.19888711950256868

Base: 0.35140039018183716
LoRA: 0.36429080506404876



In [21]:
t_stat, p_value = ttest_rel(SpearmanBase,SpearmanLoRA)
print(p_value)

0.030781020358046095


# Fine tune model using 5 epochs, learning rate 1e-3, CausalLM, num_samples = 40

In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 5
lr = 1e-3
        
model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.8, 
                                        num_samples=40, print_info=False)

print(f"Training loss = {train_losses}")
print(f"Validation loss = {val_losses}")
        

Training loss = [6.5764336585998535, 5.786240100860596, 4.082671165466309, 3.0122482776641846, 3.1828436851501465]
Validation loss = [8.93297290802002, 5.237157344818115, 3.620352268218994, 3.261817216873169, 3.030447483062744]


In [9]:
model.eval()
ft_model_name = 'llr_pg2_lora_eps5_lr1eneg3_PF00030_CLM_ns40'

In [10]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key][ft_model_name] = llr

In [11]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]][ft_model_name]

array([[-1.4291077 , -0.96684265, -2.1694262 , ..., -1.6117249 ,
         0.        , -0.02457428],
       [-1.8168182 , -1.9846036 , -2.622757  , ..., -1.9904175 ,
        -3.1830292 , -2.8275375 ],
       [-1.9161148 , -2.5897064 , -3.4197083 , ..., -0.61306   ,
        -3.861374  , -2.679184  ],
       ...,
       [-3.1298904 ,  0.        , -2.7935715 , ..., -3.5385818 ,
        -6.38797   , -3.204544  ],
       [-1.4543152 ,  0.40469354, -3.8770905 , ..., -3.0529785 ,
        -5.666115  , -3.3366773 ],
       [ 0.        , -3.6659164 , -3.4661944 , ..., -3.1143646 ,
        -6.973686  , -5.496582  ]], shape=(89, 20), dtype=float32)

In [12]:
SpearmanBase = []
SpearmanLoRA = []

for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key][ft_model_name]
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    SpearmanBase.append(spb[0])
    SpearmanLoRA.append(spl[0])

In [13]:
for i in range(len(SpearmanBase)):
    spb = SpearmanBase[i]
    spl = SpearmanLoRA[i]
    print(f"Base: {spb}")
    print(f"LoRA: {spl}")
    print()

Base: 0.4172571483248994
LoRA: 0.36802936431511024

Base: 0.33897700677632076
LoRA: 0.25334333093567807

Base: 0.36205722660833856
LoRA: 0.2132439388519887

Base: 0.21845045525531706
LoRA: 0.15894248196612654

Base: 0.3662332108928841
LoRA: 0.25052967579564217

Base: 0.2790765656023974
LoRA: 0.23502393362027466

Base: 0.43572135931376255
LoRA: 0.30488556129105854

Base: 0.31101513899232713
LoRA: 0.23625020629675084

Base: 0.26475198062516286
LoRA: 0.2574553976471904

Base: 0.3800703713136749
LoRA: 0.2651826947488685

Base: 0.184950854040034
LoRA: 0.22277523254981746

Base: 0.35140039018183716
LoRA: 0.20811710197122468



In [14]:
t_stat, p_value = ttest_rel(SpearmanBase,SpearmanLoRA)
print(p_value)

0.0006027711103973964


# Fine tune model using 5 epochs, learning rate 1e-3, CausalLM

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 5
lr = 1e-3
        
model, train_losses, val_losses = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.5, 
                                        num_samples=20, print_info=False)

print(f"Training loss = {train_losses}")
print(f"Validation loss = {val_losses}")
        

Training loss = [5.260550498962402, 2.5332024097442627, 3.372142791748047, 2.908649206161499, 1.829838752746582]
Validation loss = [3.0650134086608887, 6.4389142990112305, 3.0385022163391113, 2.2131755352020264, 2.498478412628174]


Computing LLR matrices using fine-tune model

In [8]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): ProGenForCausalLM(
      (transformer): ProGenModel(
        (wte): Embedding(32, 1536)
        (drop): Dropout(p=0.0, inplace=False)
        (h): ModuleList(
          (0-26): 27 x ProGenBlock(
            (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
            (attn): ProGenAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
              (resid_dropout): Dropout(p=0.0, inplace=False)
              (qkv_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=4608, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4608, bias=False)
      

In [9]:
ft_model_name = 'llr_pg2_lora_eps5_lr1eneg3_PF00030_CLM'

In [10]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key][ft_model_name] = llr

In [11]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


In [12]:
base_model.eval()

ProGenForCausalLM(
  (transformer): ProGenModel(
    (wte): Embedding(32, 1536)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-26): 27 x ProGenBlock(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): ProGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1536, out_features=4608, bias=False)
          (out_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): ProGenMLP(
          (fc_in): Linear(in_features=1536, out_features=6144, bias=True)
          (fc_out): Linear(in_features=6144, out_features=1536, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=32, bias=True)
)

In [14]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]][ft_model_name]

array([[-1.2624817 , -0.8487549 , -1.810982  , ..., -1.545517  ,
         0.        , -0.55914307],
       [-1.1810304 , -1.3902433 , -1.6130066 , ..., -1.4779054 ,
        -1.925705  , -1.9196777 ],
       [-1.0403748 , -1.797035  , -2.2122648 , ..., -0.33535004,
        -2.3332517 , -1.7674408 ],
       ...,
       [-2.6438522 ,  0.        , -2.9620438 , ..., -3.0952911 ,
        -5.3216705 , -3.180191  ],
       [-1.1033096 ,  0.5032043 , -3.7488785 , ..., -2.4773483 ,
        -4.601883  , -3.0935514 ],
       [ 0.        , -3.2847748 , -3.078438  , ..., -2.532837  ,
        -5.6741943 , -5.0941315 ]], shape=(89, 20), dtype=float32)

In [20]:
SpearmanBase = []
SpearmanLoRA = []

for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key][ft_model_name]
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    SpearmanBase.append(spb[0])
    SpearmanLoRA.append(spl[0])


    # print(f"Base model spearman correlation with DMS is {spb[0]}")
    # print(f"Lora model spearman correlation with DMS is {spl[0]}")
    # print()

In [22]:
t_stat, p_value = ttest_rel(SpearmanBase,SpearmanLoRA)
print(p_value)

0.5795587054215662


# Fine tune model using 50 epochs, learning rate 1e-3, CausalLM

In [10]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    # task_type=TaskType.FEATURE_EXTRACTION
    task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 50
lr = 1e-3
        
model, val_loss = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.5, 
                                        num_samples=20, print_info=False)

print(f"Validation loss = {val_loss}")
        

Validation loss = 2.073259115219116


save fine-tuned model

In [ ]:
# # save a model
# model_dir = '/Users/johnhutchens/Desktop/Practicum/Models/'+'pg2_lora_eps50_lr1eneg3_PF00030_CLM'

# model.save_pretrained(model_dir)
# tokenizer.save_pretrained(model_dir)

('/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/tokenizer_config.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/special_tokens_map.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/vocab.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/merges.txt',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/added_tokens.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps50_lr1eneg3_PF00030_CLM/tokenizer.json')

load fine-tuned model

In [ ]:
# model_dir = '/Users/johnhutchens/Desktop/Practicum/Models/'+'pg2_lora_eps30_lr1eneg3_PF00030'

# base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
# model = PeftModel.from_pretrained(base_model, model_dir)
# tokenizer = AutoTokenizer.from_pretrained(model_dir)

construct LLR matrices using fine-tuned model

In [20]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): ProGenForCausalLM(
      (transformer): ProGenModel(
        (wte): Embedding(32, 1536)
        (drop): Dropout(p=0.0, inplace=False)
        (h): ModuleList(
          (0-26): 27 x ProGenBlock(
            (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
            (attn): ProGenAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
              (resid_dropout): Dropout(p=0.0, inplace=False)
              (qkv_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=4608, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4608, bias=False)
      

In [21]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key]['llr_pg2_lora_eps50_lr1eneg3_PF00030_CLM'] = llr

reset base model (may not be necessary...)

In [22]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


construct LLR matrices using base model

In [23]:
base_model.eval()

ProGenForCausalLM(
  (transformer): ProGenModel(
    (wte): Embedding(32, 1536)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-26): 27 x ProGenBlock(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): ProGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1536, out_features=4608, bias=False)
          (out_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): ProGenMLP(
          (fc_in): Linear(in_features=1536, out_features=6144, bias=True)
          (fc_out): Linear(in_features=6144, out_features=1536, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=32, bias=True)
)

In [24]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, base_model, tokenizer)
    dict_PF00030[key]['llr_pg2'] = llr

quick check for difference in model outputs

In [42]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]]['llr_pg2_lora_eps50_lr1eneg3_PF00030_CLM']

array([[-3.5778732, -3.1686707, -3.9479983, ..., -3.6606445,  0.       ,
        -1.4397736],
       [-3.299385 , -3.742729 , -3.9519424, ..., -3.5338516, -3.8312607,
        -4.334999 ],
       [-2.810051 , -5.024132 , -4.5925217, ..., -1.2912979, -5.0251236,
        -4.335083 ],
       ...,
       [-2.1934204,  0.       , -1.445549 , ..., -2.4002   , -5.580246 ,
        -2.2692642],
       [-1.2422333,  0.3035125, -3.333397 , ..., -2.6573944, -4.5811615,
        -2.7085645],
       [ 0.       , -4.290077 , -3.11644  , ..., -3.1033401, -6.1597824,
        -5.277649 ]], shape=(89, 20), dtype=float32)

compare base vs fine tuned llr/dms spearman correlations

In [43]:
for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key]['llr_pg2_lora_eps50_lr1eneg3_PF00030_CLM']
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    print(f"Base model spearman correlation with DMS is {spb[0]}")
    print(f"Lora model spearman correlation with DMS is {spl[0]}")
    print()

Base model spearman correlation with DMS is 0.4172571483248994
Lora model spearman correlation with DMS is 0.2105412739574139

Base model spearman correlation with DMS is 0.33897700677632076
Lora model spearman correlation with DMS is 0.1750848402774465

Base model spearman correlation with DMS is 0.36205722660833856
Lora model spearman correlation with DMS is 0.10390199985565027

Base model spearman correlation with DMS is 0.21845045525531706
Lora model spearman correlation with DMS is 0.08534220455322551

Base model spearman correlation with DMS is 0.3662332108928841
Lora model spearman correlation with DMS is 0.2213490234279677

Base model spearman correlation with DMS is 0.2790765656023974
Lora model spearman correlation with DMS is 0.08154157594993876

Base model spearman correlation with DMS is 0.43572135931376255
Lora model spearman correlation with DMS is 0.17499160761901747

Base model spearman correlation with DMS is 0.31101513899232713
Lora model spearman correlation with DM

try rescaling adapter

# Fine tune model using 30 epochs, learning rate of 1e-3, FeatureExtraction

In [38]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # target_modules=["query", "key", "value", "output.dense"],
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
    # task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 30
lr = 1e-3
        
model, val_loss = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.5, 
                                        num_samples=20, print_info=False)

print(f"Validation loss = {val_loss}")
        

Validation loss = 2.1665737628936768


In [ ]:
# save a model
# model_dir = '/Users/johnhutchens/Desktop/Practicum/Models/'+'pg2_lora_eps30_lr1eneg3_PF00030'

# model.save_pretrained(model_dir)
# tokenizer.save_pretrained(model_dir)

('/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/tokenizer_config.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/special_tokens_map.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/vocab.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/merges.txt',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/added_tokens.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/tokenizer.json')

In [70]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key]['llr_pg2_lora_eps30_lr1eneg3_PF00030'] = llr

In [71]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, base_model, tokenizer)
    dict_PF00030[key]['llr_pg2'] = llr

In [72]:
keys = list(dict_PF00030.keys())

In [73]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]]['llr_pg2_lora_eps30_lr1eneg3_PF00030']

array([[-2.59404   , -2.0913162 , -3.4876025 , ..., -2.593216  ,
         0.        , -0.77513885],
       [-3.553444  , -3.9058225 , -4.533386  , ..., -3.706253  ,
        -4.620201  , -4.8306885 ],
       [-2.7412338 , -4.2763214 , -5.300476  , ..., -1.0279007 ,
        -5.292282  , -4.0886993 ],
       ...,
       [-2.5057983 ,  0.        , -2.6578522 , ..., -2.8506927 ,
        -6.108902  , -2.8343887 ],
       [-1.4111786 ,  0.2569732 , -3.8256226 , ..., -2.8599472 ,
        -5.49057   , -3.2911375 ],
       [ 0.        , -4.3262253 , -3.6228716 , ..., -3.092659  ,
        -7.2896194 , -5.860054  ]], shape=(89, 20), dtype=float32)

In [ ]:
# path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'
# with open(path+"dict_PF00030.pkl", "wb") as f:
#     pickle.dump(dict_PF00030, f)

In [80]:
for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key]['llr_pg2_lora_eps30_lr1eneg3_PF00030']
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    print(f"Base model spearman correlation with DMS is {spb[0]}")
    print(f"Lora model spearman correlation with DMS is {spl[0]}")
    print()

Base model spearman correlation with DMS is 0.4172571483248994
Lora model spearman correlation with DMS is 0.15932049492101447

Base model spearman correlation with DMS is 0.33897700677632076
Lora model spearman correlation with DMS is 0.17179156530206946

Base model spearman correlation with DMS is 0.36205722660833856
Lora model spearman correlation with DMS is 0.12497598169493365

Base model spearman correlation with DMS is 0.21845045525531706
Lora model spearman correlation with DMS is 0.16575524324784763

Base model spearman correlation with DMS is 0.3662332108928841
Lora model spearman correlation with DMS is 0.05460478145640364

Base model spearman correlation with DMS is 0.2790765656023974
Lora model spearman correlation with DMS is 0.07376780266484208

Base model spearman correlation with DMS is 0.43572135931376255
Lora model spearman correlation with DMS is 0.08960826853333739

Base model spearman correlation with DMS is 0.31101513899232713
Lora model spearman correlation with

In [50]:
dict_PF00030[keys[2]]['DMS_mat'] - dict_PF00030[keys[3]]['DMS_mat']

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(89, 20))